# Task 1: Install required Python libraries

In [68]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: /Users/snigdhatiwari/ai-football/.venv/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
3.0.3
1.9.0
1.58.0
1.5.3
2.34.2


# Task 2: Download the dataset

In [69]:
from pathlib import Path
import requests
import pandas as pd

# Create data/ folder if it doesn't exist
Path("data").mkdir(exist_ok=True)

# Download results.csv only if not already present
csv_path = Path("data/results.csv")
if not csv_path.exists():
    url = (
        "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge"
        "/hands-on-labs/main/02_football_lab_june/04_data/results.csv"
    )
    response = requests.get(url)
    response.raise_for_status()
    csv_path.write_bytes(response.content)

# Load into a DataFrame
matches = pd.read_csv(csv_path, parse_dates=["date"])

print(matches.shape)
print(matches["date"].min())
print(matches["date"].max())
matches.head(3)

(49329, 9)
1872-11-30 00:00:00
2026-06-27 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


# Task 3: Explore the data

In [70]:
import pandas as pd

# --- Top 10 most frequent tournaments ---
print("Top 10 tournaments by match count:")
print(matches["tournament"].value_counts().head(10).to_string())

# --- Top 15 teams by total appearances (home + away) ---
print("\nTop 15 teams by total matches played:")
team_counts = (
    pd.concat([matches["home_team"], matches["away_team"]])
    .value_counts()
    .head(15)
)
print(team_counts.to_string())

# --- Matches per decade ---
print("\nMatches per decade:")
decade = (matches["date"].dt.year // 10 * 10).rename("decade")
decade_counts = decade.value_counts().sort_index()
decade_counts.index = decade_counts.index.astype(str).str.cat(["s"] * len(decade_counts))
print(decade_counts.to_string())

Top 10 tournaments by match count:
tournament
Friendly                                18257
Soccer World Cup qualification           8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
Soccer World Cup                         1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620

Top 15 teams by total matches played:
Sweden         1102
England        1091
Argentina      1069
Brazil         1060
Germany        1032
South Korea    1008
Hungary        1004
Mexico         1003
Uruguay         973
France          936
Italy           891
Poland          890
Switzerland     885
Netherlands     880
Norway          873

Matches per decade:
decade
1870s      13
1880s      55
1890s      59
1900s     137
1910s     330
1920s     831
1930s    1079
1940s     833
1950s    1651
1

# Task 4: Engineer features for the model

In [71]:
from collections import defaultdict
import pandas as pd

# --- Filter and sort ---
df = (
    matches[matches["date"] >= "1990-01-01"]
    .sort_values("date")
    .reset_index(drop=True)
)

MAJOR_TOURNAMENTS = {
    "Soccer World Cup",
    "Soccer World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
}

# --- Helper functions ---
def winrate(hist):
    """Win rate over all prior matches; default 0.5 if no history."""
    if not hist:
        return 0.5
    return sum(h[2] for h in hist) / len(hist)

def goal_avg(hist):
    """Average goals scored over all prior matches; default 1.0 if no history."""
    if not hist:
        return 1.0
    return sum(h[0] for h in hist) / len(hist)

def recent_form(hist):
    """Win rate over last 10 matches; default 0.5 if fewer than 10 prior matches."""
    if len(hist) < 10:
        return 0.5
    last10 = hist[-10:]
    return sum(h[2] for h in last10) / 10

# --- Build features row by row ---
# team_history maps team name -> list of (goals_for, goals_against, won)
team_history = defaultdict(list)
rows = []

for _, row in df.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    hist_home = team_history[home]
    hist_away = team_history[away]

    # Compute features from history BEFORE this match
    features = {
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "team_a_winrate": winrate(hist_home),
        "team_b_winrate": winrate(hist_away),
        "team_a_goal_avg": goal_avg(hist_home),
        "team_b_goal_avg": goal_avg(hist_away),
        "team_a_recent_form": recent_form(hist_home),
        "team_b_recent_form": recent_form(hist_away),
        "is_neutral": int(row["neutral"]),
        "is_major_tournament": int(row["tournament"] in MAJOR_TOURNAMENTS),
    }

    # Determine outcome
    hs, as_ = row["home_score"], row["away_score"]
    if hs > as_:
        features["outcome"] = 0   # home win
    elif hs == as_:
        features["outcome"] = 1   # draw
    else:
        features["outcome"] = 2   # away win

    rows.append(features)

    # Update history AFTER computing features (prevents leakage)
    home_won = int(hs > as_)
    away_won = int(as_ > hs)
    team_history[home].append((hs, as_, home_won))
    team_history[away].append((as_, hs, away_won))

features_df = pd.DataFrame(rows)

print(features_df.shape)
features_df.head(3)

(32212, 12)


,date,home_team,away_team,team_a_winrate,team_b_winrate,team_a_goal_avg,team_b_goal_avg,team_a_recent_form,team_b_recent_form,is_neutral,is_major_tournament,outcome
0,1990-01-12,Algeria,Mali,0.5,0.5,1.0,1.0,0.5,0.5,1,0,0
1,1990-01-14,Algeria,Cameroon,1.0,0.5,5.0,1.0,0.5,0.5,1,0,0
2,1990-01-17,Greece,Belgium,0.5,0.5,1.0,1.0,0.5,0.5,0,0,0


# Task 5: Split data into training and test sets

In [72]:
feature_cols = [
    "team_a_winrate",
    "team_b_winrate",
    "team_a_goal_avg",
    "team_b_goal_avg",
    "team_a_recent_form",
    "team_b_recent_form",
    "is_neutral",
    "is_major_tournament",
]

# Time-based split: train < 2018-01-01, test >= 2018-01-01
train_mask = features_df["date"] < "2018-01-01"
test_mask  = features_df["date"] >= "2018-01-01"

X_train = features_df.loc[train_mask, feature_cols]
X_test  = features_df.loc[test_mask,  feature_cols]
y_train = features_df.loc[train_mask, "outcome"]
y_test  = features_df.loc[test_mask,  "outcome"]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("\ny_train class distribution:")
print(y_train.value_counts(normalize=True).rename({0: 'home win', 1: 'draw', 2: 'away win'}).to_string())

X_train: (24179, 8)
X_test:  (8033, 8)
y_train: (24179,)
y_test:  (8033,)

y_train class distribution:
outcome
home win    0.487117
away win    0.276273
draw        0.236610


# Task 6: Train and evaluate the model

In [73]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

LABELS = ["Home win", "Draw", "Away win"]

# --- Train ---
model = RandomForestClassifier(
    n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# --- Test accuracy ---
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy:  {test_acc * 100:.2f}%")

# --- Baseline: always predict most frequent class in y_train ---
most_frequent = y_train.value_counts().idxmax()
baseline_acc = accuracy_score(y_test, [most_frequent] * len(y_test))
print(f"Baseline accuracy (always predict class {most_frequent} – {LABELS[most_frequent]}): {baseline_acc * 100:.2f}%")

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f"Actual: {l}" for l in LABELS],
                         columns=[f"Pred: {l}" for l in LABELS])
print("\nConfusion matrix:")
print(cm_df.to_string())

# --- Feature importances ---
print("\nFeature importances (descending):")
importances = sorted(zip(feature_cols, model.feature_importances_),
                     key=lambda x: x[1], reverse=True)
for name, score in importances:
    print(f"  {name:<28} {score:.4f}")

Test accuracy:  55.98%
Baseline accuracy (always predict class 0 – Home win): 47.19%

Confusion matrix:
                  Pred: Home win  Pred: Draw  Pred: Away win
Actual: Home win            3393          34             364
Actual: Draw                1390          29             419
Actual: Away win            1277          52            1075

Feature importances (descending):
  team_b_winrate               0.2203
  team_a_winrate               0.2062
  team_b_goal_avg              0.1889
  team_a_goal_avg              0.1821
  team_b_recent_form           0.0752
  team_a_recent_form           0.0750
  is_neutral                   0.0275
  is_major_tournament          0.0248


# Task 7: Save the model and team statistics

In [74]:
from pathlib import Path
import pandas as pd
import joblib

# --- Create models/ directory ---
Path("models").mkdir(exist_ok=True)

# --- Build set of World Cup-eligible teams ---
wc_qual = matches[matches["tournament"] == "Soccer World Cup qualification"]
soccer_teams = set(wc_qual["home_team"]).union(set(wc_qual["away_team"]))

# --- Build team_stats ---
# Collect per-team match rows: (date, goals_for, won)
from collections import defaultdict

team_records = defaultdict(list)  # team -> list of (date, goals_for, won)

for _, row in matches.iterrows():
    home, away = row["home_team"], row["away_team"]
    hs, as_ = row["home_score"], row["away_score"]
    date = row["date"]
    home_won = int(hs > as_)
    away_won = int(as_ > hs)
    team_records[home].append((date, hs, home_won))
    team_records[away].append((date, as_, away_won))

team_stats = {}
for team, records in team_records.items():
    # Skip non-Soccer-World-Cup-eligible teams
    if team not in soccer_teams:
        continue
    total = len(records)
    if total < 30:
        continue
    # Sort by date to get chronological order for recent_form
    records_sorted = sorted(records, key=lambda x: x[0])
    wins = sum(r[2] for r in records_sorted)
    goals = sum(r[1] for r in records_sorted)
    last10 = records_sorted[-10:]
    rf = sum(r[2] for r in last10) / 10 if total >= 10 else 0.5
    team_stats[team] = {
        "winrate": wins / total,
        "goal_avg": goals / total,
        "recent_form": rf,
        "matches_played": total,
    }

# --- Save model and team data ---
joblib.dump(model, "models/match_predictor.pkl")
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols}, "models/team_data.pkl")

# --- Summary ---
print(f"Teams stored: {len(team_stats)}")
print("\nTop 5 teams by win rate (≥ 100 matches):")
stats_df = pd.DataFrame.from_dict(team_stats, orient="index")
top5 = (
    stats_df[stats_df["matches_played"] >= 100]
    .sort_values("winrate", ascending=False)
    .head(5)[["winrate", "goal_avg", "recent_form", "matches_played"]]
)
print(top5.to_string())

Teams stored: 215

Top 5 teams by win rate (≥ 100 matches):
          winrate  goal_avg  recent_form  matches_played
Brazil   0.632075       NaN          0.3            1060
Spain    0.586735       NaN          0.5             784
Germany  0.578488       NaN          0.7            1032
England  0.571036       NaN          0.5            1091
Iran     0.566069       NaN          0.3             613


# Task 8: Try the model

In [75]:
import pandas as pd

def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    """Predict match outcome probabilities between two teams."""
    if team_a not in team_stats:
        raise ValueError(
            f"'{team_a}' not found in team_stats. "
            f"Check spelling or try a different team name."
        )
    if team_b not in team_stats:
        raise ValueError(
            f"'{team_b}' not found in team_stats. "
            f"Check spelling or try a different team name."
        )

    stats_a = team_stats[team_a]
    stats_b = team_stats[team_b]

    row = pd.DataFrame([{
        "team_a_winrate":      stats_a["winrate"],
        "team_b_winrate":      stats_b["winrate"],
        "team_a_goal_avg":     stats_a["goal_avg"],
        "team_b_goal_avg":     stats_b["goal_avg"],
        "team_a_recent_form":  stats_a["recent_form"],
        "team_b_recent_form":  stats_b["recent_form"],
        "is_neutral":          int(is_neutral),
        "is_major_tournament": int(is_major_tournament),
    }]).reindex(columns=feature_cols)

    proba = model.predict_proba(row)
    return {
        "team_a_win_prob": round(float(proba[0][0]), 4),
        "draw_prob":       round(float(proba[0][1]), 4),
        "team_b_win_prob": round(float(proba[0][2]), 4),
    }


result1 = predict_match("Brazil", "Argentina")
print(f"Brazil vs Argentina:  {result1}")

result2 = predict_match("Germany", "Brazil")
print(f"Germany vs Brazil:    {result2}")

Brazil vs Argentina:  {'team_a_win_prob': 0.3299, 'draw_prob': 0.3167, 'team_b_win_prob': 0.3534}
Germany vs Brazil:    {'team_a_win_prob': 0.5104, 'draw_prob': 0.218, 'team_b_win_prob': 0.2716}


# Task 9: Create an interactive UI application

In [76]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

st.set_page_config(
    page_title="Soccer 2026 Match Predictor",
    page_icon="⚽",
    layout="centered",
)


@st.cache_resource
def load_artifacts():
    model = joblib.load("models/match_predictor.pkl")
    team_data = joblib.load("models/team_data.pkl")
    return model, team_data["team_stats"], team_data["feature_cols"]


model, team_stats, feature_cols = load_artifacts()

st.title("⚽ Soccer 2026 Match Predictor")
st.caption("Predictions based on historical international football results (1872–2026).")

team_names = sorted(team_stats.keys())

col1, col2 = st.columns(2)
with col1:
    default_a = team_names.index("Brazil") if "Brazil" in team_names else 0
    team_a = st.selectbox("Team A", team_names, index=default_a)
with col2:
    default_b = team_names.index("Argentina") if "Argentina" in team_names else 1
    team_b = st.selectbox("Team B", team_names, index=default_b)

is_neutral = st.checkbox("Neutral venue", value=True)
is_major   = st.checkbox("Major tournament (e.g. World Cup)", value=True)

if st.button("Predict", type="primary", use_container_width=True):
    if team_a == team_b:
        st.error("Please pick two different teams.")
    else:
        stats_a = team_stats[team_a]
        stats_b = team_stats[team_b]

        row = pd.DataFrame([{
            "team_a_winrate":      stats_a["winrate"],
            "team_b_winrate":      stats_b["winrate"],
            "team_a_goal_avg":     stats_a["goal_avg"],
            "team_b_goal_avg":     stats_b["goal_avg"],
            "team_a_recent_form":  stats_a["recent_form"],
            "team_b_recent_form":  stats_b["recent_form"],
            "is_neutral":          int(is_neutral),
            "is_major_tournament": int(is_major),
        }]).reindex(columns=feature_cols)

        proba = model.predict_proba(row)[0]
        p_a, p_draw, p_b = float(proba[0]), float(proba[1]), float(proba[2])

        # --- Metrics ---
        m1, m2, m3 = st.columns(3)
        m1.metric(f"{team_a} wins", f"{p_a * 100:.1f}%")
        m2.metric("Draw",           f"{p_draw * 100:.1f}%")
        m3.metric(f"{team_b} wins", f"{p_b * 100:.1f}%")

        # --- Progress bars ---
        st.progress(p_a,    text=f"{team_a} wins")
        st.progress(p_draw, text="Draw")
        st.progress(p_b,    text=f"{team_b} wins")

        # --- Team stats table ---
        st.table(
            pd.DataFrame(
                {
                    "Win rate":            [round(stats_a["winrate"],      3), round(stats_b["winrate"],      3)],
                    "Avg goals scored":    [round(stats_a["goal_avg"],     3), round(stats_b["goal_avg"],     3)],
                    "Recent form (last 10)":[round(stats_a["recent_form"], 3), round(stats_b["recent_form"], 3)],
                    "Matches played":      [stats_a["matches_played"],        stats_b["matches_played"]],
                },
                index=[team_a, team_b],
            )
        )

Overwriting app.py


# Task 10: Launch the UI application

In [77]:
import subprocess
import sys
import time
import webbrowser

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.headless", "true",
        "--server.port", "8501",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Starting Streamlit server...")
time.sleep(4)

webbrowser.open("http://localhost:8501")

print("Streamlit app is running at: http://localhost:8501")
print("To stop the server, run: streamlit_process.terminate()")

Starting Streamlit server...
Streamlit app is running at: http://localhost:8501
To stop the server, run: streamlit_process.terminate()
